# De-Id SLM — v3 on Colab (4-bit QLoRA via Unsloth)

Reproduces the **v3** name-judgment fine-tune on a Colab **GPU (T4)** runtime, using the real
4-bit QLoRA path (`unsloth`) — the assignment's actual target backend. The Mac runs (v2/v3) used
plain LoRA on a bf16 base; **this run re-baselines** because it 4-bit-quantizes the base, so report
these Colab numbers as their own line — do **not** reuse the MPS numbers from `docs/results.md`.

**Runtime → Change runtime type → GPU (T4)** before running. No teacher API key is required:
the v3 data is either the committed splits (default) or regenerated locally via the in-session
`--provider authored` teacher (`src/datagen/author.py`).

Hard ceiling honored: the leakage check + eval-before-training gate run below, before any training.

## 0. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — set Runtime type to GPU (T4)'

## 1. Clone the repo
Set `BRANCH` to the branch that has the **v3 data committed** (`data/splits/*.jsonl` + `src/datagen/author.py`).
As of writing that work lives on `agent/datagen-v2-run` and is not yet merged to `main`.

In [ ]:
BRANCH = 'agent/datagen-v2-run'  # change to 'main' once v3 is merged
!git clone -q -b $BRANCH https://github.com/f15cubing/slm-deid.git
%cd slm-deid
!git log --oneline -1

## 2. Install deps
On Linux this pulls the CUDA stack (`unsloth` + `bitsandbytes`), which are gated out on macOS.

In [ ]:
!pip install -q -r requirements.txt
# If Colab's preinstalled torch conflicts with unsloth, restart the runtime once after this cell.

## 3. Data
**Mode A (default):** use the committed v3 splits (`data/splits/`) — includes the CRAPII real slice.
**Mode B (optional, fully self-contained):** regenerate **synthetic-only** v3 from templates (no API,
no CRAPII) — run the Mode B cell instead if the committed data is stale or you want a clean rebuild.
Either way, `configs/train.yaml` reads `data/splits/{train,val}.jsonl`.

In [ ]:
# Mode A — inspect the committed v3 splits
import json, collections
from src.common import tags
rows=[json.loads(l) for l in open('data/splits/train.jsonl')]
tag=sum(1 for r in rows if tags.NAME_OPEN in r['target'])
print(f'train={len(rows)} (tagged {tag} / untagged {len(rows)-tag})')
print('sources:', dict(collections.Counter(r['source'] for r in rows)))
print('categories:', dict(collections.Counter(r['category'] for r in rows).most_common()))

In [ ]:
# Mode B (OPTIONAL) — regenerate synthetic-only v3 (no API key, no CRAPII). Uncomment to use.
# !python -m src.datagen.generate --config configs/datagen.yaml --provider authored --seed 1 --out-dir data/_b1
# !python -m src.datagen.merge --out data --sources data/_b1/splits/train.jsonl data/_b1/splits/val.jsonl
# (No --crapii: the 109MB CRAPII raw file is gitignored and absent on Colab, so Mode B is synthetic-only.)

## 4. Hard-ceiling gate — eval exists + no eval leakage
Must pass before any training run (per `AGENTS.md`).

In [ ]:
!python -m pytest tests/test_no_eval_leakage.py tests/test_vocab.py -q
# Independent leakage re-check on the actual splits (all three guards must be 0):
from pathlib import Path
from src.common.schema import read_jsonl
from src.datagen.generate import _norm, _eval_inputs, drop_eval_token_overlap, drop_eval_surface_overlap
allex=list(read_jsonl('data/splits/train.jsonl'))+list(read_jsonl('data/splits/val.jsonl'))
ev=_eval_inputs('eval')
g1=sum(1 for e in allex if _norm(e.input) in ev)
_,g2=drop_eval_token_overlap(allex,'eval'); _,g3=drop_eval_surface_overlap(allex)
print(f'leakage guards — exact {g1}  token {g2}  surface {g3}  (all must be 0)')
assert g1==g2==g3==0, 'EVAL LEAKAGE DETECTED — do not train'

## 5. Confirm the CUDA/unsloth backend auto-selects

In [ ]:
from src.common.device import detect_backend, pick_device
print('backend:', detect_backend(), '| device:', pick_device())  # expect: unsloth | cuda

## 6. Train — 4-bit QLoRA (v3 data, frozen hyperparameters)
Same LoRA recipe as the Mac runs (r=32, α=32, lr=2e-4, 3 epochs, completion-only); only the backend
(4-bit unsloth) differs. ~a few minutes on a T4.

In [ ]:
!python -m src.train.qlora --config configs/train.yaml --output-dir outputs/sft-v3-colab

## 7. Eval — base vs tuned on the 51 quarantined hard cases
Both models run the same backend; F5 (recall-weighted) is the headline.

In [ ]:
!python -m src.eval.run --split eval/hardcases --compare base outputs/sft-v3-colab --report-dir outputs/eval_reports

In [ ]:
# Render the overall + per-category tables with 95% bootstrap CIs (offline; no model/network).
# Uses the two reports just written (globs match one each on a fresh run):
!python -m src.eval.report base=outputs/eval_reports/base-*.json tuned=outputs/eval_reports/tuned-*.json

## 8. Save the adapter + pin versions
Colab is ephemeral — download the adapter (or mount Drive). Also capture the exact GPU env to pin
`requirements.txt` (still a TODO in the repo).

In [ ]:
!pip freeze | grep -Ei 'unsloth|bitsandbytes|torch|transformers|trl|peft|accelerate' > outputs/sft-v3-colab/pip_freeze_gpu.txt
!cat outputs/sft-v3-colab/pip_freeze_gpu.txt
import shutil; shutil.make_archive('sft-v3-colab', 'zip', 'outputs/sft-v3-colab')
try:
    from google.colab import files; files.download('sft-v3-colab.zip')
except Exception as e:
    print('download skipped:', e)

### Notes
- **Re-baseline:** these Colab 4-bit numbers are a separate line from the Mac bf16 v3 numbers — add a
  `v3-colab` row to `docs/results.md`, don't overwrite the MPS one.
- **Pin `requirements.txt`** from `pip_freeze_gpu.txt` above (the repo's long-standing TODO).
- **Known limitation (deferred):** on messy real text the byte-identity output format breaks (whitespace
  normalization); the span-offset fix is tracked in `docs/STATUS.md` backlog — not exercised here.